# NJ + NY commute weights

Same result as Weikun's commute notebooks, rebuilt in this folder from the Census source files.

**Downloaded into `Data/LODES`**
- [LODES 2021 OD](https://lehd.ces.census.gov/data/lodes/LODES8/): NJ and NY main files (home and work in the same state) and aux files (home and work in different states)
- NJ and NY block-to-ZCTA crosswalks
- [2021 ZCTA Gazetteer](https://www2.census.gov/geo/docs/maps-data/data/gazetteer/2021_Gazetteer/2021_Gaz_zcta_national.zip) for land area, saved under `Data/Census`

**Population**
The Census API now requires a free key, so the 2021 ACS population table already in the project (`Data/Census/njny_zcta_population_density_2021.csv`, ACS table B01003) is the denominator for case rates.

**What `build_njny_commute.py` wrote**
- `njny_commute_outputs/njny_commute_zcta_flows_2021.csv` — jobs from home ZCTA to work ZCTA
- `njny_commute_zcta_weights_2021.csv` — each outside link's share of a workplace's incoming jobs
- `njny_commute_zcta_features_2021.csv` — one summary row per ZCTA
- an audit whose job totals match Weikun's: NJ→NJ 3,439,824; NY→NY 7,897,753; NJ→NY 494,304; NY→NJ 187,042

Run the next cells from the project root. They only read the saved tables.


In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path("Dhruvesh")
audit = pd.read_csv(ROOT / "njny_commute_outputs" / "njny_commute_processing_audit_2021.csv")
audit


## One workplace

People who live and work in the same ZCTA are left out of the weights. Each remaining link's `inbound_weight` is its share of the jobs coming in from somewhere else. Those shares sum to 1.


In [ ]:
flows = pd.read_csv(
    ROOT / "njny_commute_outputs" / "njny_commute_zcta_flows_2021.csv",
    dtype={"origin_zcta": str, "destination_zcta": str},
)
weights = pd.read_csv(
    ROOT / "njny_commute_outputs" / "njny_commute_zcta_weights_2021.csv",
    dtype={"origin_zcta": str, "destination_zcta": str},
)

dest_state, dest_zcta = "NJ", "07302"
into = flows.loc[flows.destination_state.eq(dest_state) & flows.destination_zcta.eq(dest_zcta)]
internal = into.loc[into.origin_state.eq(dest_state) & into.origin_zcta.eq(dest_zcta), "commute_jobs"].sum()
outside = weights.loc[weights.destination_state.eq(dest_state) & weights.destination_zcta.eq(dest_zcta)]
print(f"{dest_state} {dest_zcta}: {int(internal):,} jobs live and work here")
print(f"Outside jobs: {int(outside.commute_jobs.sum()):,}; inbound weights sum to {outside.inbound_weight.sum():.3f}")
print(f"Of those outside jobs, from NY: {int(outside.loc[outside.origin_state.eq('NY'), 'commute_jobs'].sum()):,}")
outside.nlargest(8, "inbound_weight")[
    ["origin_state", "origin_zcta", "commute_jobs", "inbound_weight"]
]


## Case rate × weight

This uses Weikun's synthetic case file (eight New Jersey ZCTAs only) and the ACS population table. A New York origin stays in the commute table, but it does not enter the pressure number unless that origin has cases. Missing origins are not filled with zero, and the remaining weights are not rescaled.


In [ ]:
cases = pd.read_csv(
    Path("Weikun/cleaning_outputs/iphd_cleaned_records.csv"),
    dtype={"ZIP": str, "STATE": str},
)
population = pd.read_csv(
    ROOT / "Data/Census/njny_zcta_population_density_2021.csv",
    dtype={"zcta": str},
)
cases["zcta"] = cases["ZIP"].str.strip().str.zfill(5)
cases["state"] = cases["STATE"].str.upper().str.strip()
cases["date"] = pd.to_datetime(cases["ILLNESS_ONSET_DATE"])
day = pd.Timestamp("2022-01-10")
counts = (
    cases.loc[cases.date.eq(day)]
    .groupby(["state", "zcta"])
    .size()
    .rename("cases")
    .reset_index()
)
counts = counts.merge(population[["zcta", "population"]], on="zcta", how="left")
counts["case_rate_per_100k"] = counts["cases"] / counts["population"] * 100_000

known = outside.merge(
    counts,
    left_on=["origin_state", "origin_zcta"],
    right_on=["state", "zcta"],
    how="inner",
)
coverage = known["inbound_weight"].sum()
partial = (known["inbound_weight"] * known["case_rate_per_100k"]).sum()
print(f"Destination {dest_state} {dest_zcta} on {day.date()}")
print(f"Share of outside jobs whose home rate is known: {coverage:.3f}")
print(f"Partial pressure: {partial:.3f}")
print("Full pressure: blank, because coverage is not 1")
known[["origin_state", "origin_zcta", "cases", "case_rate_per_100k", "inbound_weight"]]
